In [ ]:
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "jupytergis==0.16.8",
#     "openeo",
# ]
# ///

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run lava_flow_sentinel2.ipynb
```

(The `uvx` command is included when installing `uv`).

## License

This notebook is an adaptation of the openEO-UDP notebook
[lava_visualization.ipynb](https://github.com/developmentseed/openeo-udp/blob/main/notebooks/sentinel/sentinel-2/fire_and_disaster_monitoring/lava_visualization.ipynb)
by Development Seed. That notebook adapts the Sentinel Hub wildfire visualization evalscript to volcanic lava flows and is licensed under **CC-BY-SA-4.0**.
This adaptation is also licensed under **CC-BY-SA-4.0**.

- Original evalscript: https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/markuse_fire/ (Markuse, P.)
- Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
- Conversion: Development Seed (openEO-UDP project)

# Lava flow visualization of the Cumbre Vieja eruption, La Palma, October 2021

This notebook visualizes the volcanic lava flows of the 2021 Cumbre Vieja eruption (La Palma, Canary Islands) from Sentinel-2 L2A data with openEO.
It shows the result as a dynamic tile layer in a JupyterGIS map, on top of an OpenStreetMap basemap.

The visualization uses SWIR Band 12 (2202.4 nm), which is highly sensitive to thermal emission from molten lava.
Active lava flows produce extremely high radiance in this band, which makes it ideal for detecting and mapping volcanic activity.

The steps are:
1. Connect to the openEO backend.
2. Load the Sentinel-2 L2A scene of La Palma from 10 October 2021 and make a first-valid-pixel mosaic.
3. Build a base map that blends a SWIR "urban" composite with true color, and highlight water bodies with NDVI and NDWI.
4. Encode lava intensity (from cool crust to active molten lava) with B12 thresholds as a continuous heatmap on top of the base map.
5. Show the result in JupyterGIS.

The source notebook also has binary lava styles (red or yellow mask) and a context-preserving style.
This notebook uses only the continuous heatmap style.

## Connect to the openEO backend

In [ ]:
import openeo
from openeo.processes import array_create, if_, and_, process

connection = openeo.connect("https://openeo.mpc.ds.io/")

In [ ]:
connection.authenticate_oidc()

## Parameters

The bounding box covers the main lava field of the Cumbre Vieja eruption, on the west flank of La Palma.

The time range is 10 October 2021. The end date is exclusive in openEO.
The cloud cover filter keeps only scenes with 20 % cloud cover or less.

In [ ]:
location_name = "Cumbre Vieja, La Palma, Spain"

bounding_box = {"west": -17.94, "south": 28.58, "east": -17.86, "north": 28.64}
temporal_extent = ["2021-10-10", "2021-10-11"]
max_cloud_cover = 20

# Sentinel-2 bands for the algorithm, in this order:
# B02 (blue), B03 (green), B04 (red), B08 (NIR), B11 (SWIR 1610 nm), B12 (SWIR 2190 nm)
bands = ["B02", "B03", "B04", "B08", "B11", "B12"]

## Load the Sentinel-2 data

The algorithm needs several bands: the visible and NIR bands for the base map (land cover and water), and the SWIR bands for the urban composite and the lava detection.

- **B02**: Blue, 492.4 nm
- **B03**: Green, 559.8 nm
- **B04**: Red, 664.6 nm
- **B08**: NIR, 832.8 nm
- **B11**: SWIR, 1613.7 nm
- **B12**: SWIR, 2202.4 nm

In [ ]:
s2cube = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=bounding_box,
    temporal_extent=temporal_extent,
    bands=bands,
    properties={"eo:cloud_cover": lambda x: x <= max_cloud_cover},
)

s2cube = s2cube.process(
    "apply_pixel_selection",
    pixel_selection="first",
    data=s2cube,
)

## Lava flow visualization

The lava layer is drawn on top of a base map that shows land cover such as soil, built-up areas, vegetation and water.
The base map is a blend of 30 % SWIR urban composite (B12, B11, B04) and 70 % true color. Water pixels (NDVI < -0.15, NDWI > 0.15, B04 < 0.2) get a blue tint.

Lava intensity is encoded with B12 reflectance thresholds:

| Class | B12 reflectance | Color |
|---|---|---|
| Active molten lava | > 0.8 | Bright yellow |
| Hot lava | 0.5 – 0.8 | Orange-yellow |
| Cooling lava | 0.3 – 0.5 | Orange |
| Cool crust | 0.2 – 0.3 | Red |

The red channel is boosted for all lava pixels. The green channel is boosted by an amount that depends on the intensity class, so hotter lava looks more yellow.

In [ ]:
# Lava Flow Visualization using SWIR Band 12
def lava_visualization(data):
    """
    Visualize volcanic lava flows using Sentinel-2 SWIR imagery.
    Input data array: [B02, B03, B04, B08, B11, B12]

    Uses SWIR Band 12 (2202.4 nm) which is highly sensitive to thermal
    emission from molten lava. Active lava flows produce extremely high
    radiance in this band, making it ideal for detecting and mapping
    volcanic activity.

    Styles:
    - Continuous heatmap where lava intensity is encoded using B12 thresholds
    - Active molten lava appears in bright yellow
    - Hot lava in orange-yellow
    - Cooling lava in orange
    - Cool crust in red
    - Base map preserves soil, built-up areas, vegetation, and water
    """
    B02, B03, B04, B08, B11, B12 = (
        data[0],
        data[1],
        data[2],
        data[3],
        data[4],
        data[5],
    )

    # --- Parameters ---
    brightness = 1.0
    offset = 0.0
    lava_threshold = [0.8, 0.5, 0.3, 0.2]  # B12 reflectance thresholds
    lava_sensitivity = 1.0
    boost = 1

    # --- Vegetation and water indices ---
    NDVI = (B08 - B04) / (B08 + B04)
    NDWI = (B03 - B08) / (B03 + B08)

    # --- Base map layers ---
    # True color
    tc_r = B04 * brightness * 3 + offset
    tc_g = B03 * brightness * 3 + offset
    tc_b = B02 * brightness * 3 + offset

    # Urban/SWIR composite for better terrain context
    urban_r = B12 * brightness * 3.5 + offset
    urban_g = B11 * brightness * 3.5 + offset
    urban_b = B04 * brightness * 3 + offset

    # Blend: mix urban SWIR and true color for contextual base
    base_r = urban_r * 0.3 + tc_r * 0.7
    base_g = urban_g * 0.3 + tc_g * 0.7
    base_b = urban_b * 0.3 + tc_b * 0.7

    # Water highlighting (blue tint for water bodies)
    water_detection = and_(and_(NDVI < -0.15, NDWI > 0.15), B04 < 0.2)
    base_g = if_(water_detection, base_g * 1.2 + 0.1, base_g)
    base_b = if_(water_detection, base_b * 1.5 + 0.2, base_b)

    # --- Lava detection and intensity mapping ---
    lava_intensity = B12

    # Intensity classes using B12 thresholds
    mask_0 = lava_intensity > (lava_threshold[0] / lava_sensitivity)  # Active molten

    mask_1 = and_(
        lava_intensity > (lava_threshold[1] / lava_sensitivity),
        lava_intensity <= (lava_threshold[0] / lava_sensitivity),
    )  # Hot lava

    mask_2 = and_(
        lava_intensity > (lava_threshold[2] / lava_sensitivity),
        lava_intensity <= (lava_threshold[1] / lava_sensitivity),
    )  # Cooling lava

    mask_3 = and_(
        lava_intensity > (lava_threshold[3] / lava_sensitivity),
        lava_intensity <= (lava_threshold[2] / lava_sensitivity),
    )  # Cool crust

    # Any lava present
    mask_any = lava_intensity > (lava_threshold[3] / lava_sensitivity)

    # Red channel: boosted for all lava
    r = base_r + (boost * 0.6 * B12 * mask_any)

    # Green channel: weighted by lava intensity classes
    g = base_g + (
        boost
        * B12
        * (
            0.5 * mask_0  # Active molten lava - bright yellow
            + 0.3 * mask_1  # Hot lava - orange-yellow
            + 0.15 * mask_2  # Cooling lava - orange
            + 0.0 * mask_3  # Cool crust - red only
        )
    )

    # Blue channel: unchanged (no blue in lava)
    b = base_b

    return array_create([r, g, b])


# Apply lava visualization on the data cube on the bands dimension
lava_image = s2cube.apply_dimension(
    dimension=s2cube.metadata.band_dimension.name, process=lava_visualization
)

# Linear scale to 0-255 for RGB visualization
lava_image = lava_image.linear_scale_range(
    input_min=0, input_max=1, output_min=0, output_max=255
)

graph = lava_image

## Show the result in JupyterGIS

The map has an OpenStreetMap basemap. The openEO layer is added on top of it.
`add_openeo_tile_layer` creates an XYZ service on the backend and fits the map to the bounding box.

The backend calculates each tile on request. A tile can take 10 to 25 seconds, so the layer fills in slowly.

Yellow is active molten lava, orange-yellow is hot lava, orange is cooling lava and red is cool crust.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=28.63, longitude=-17.9, zoom=11)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors",
)

In [ ]:
doc.add_openeo_tile_layer(graph, name="Lava Flow Visualization - Intensity Heatmap")
doc

## Technical notes

**Why SWIR Band 12 for lava detection?**

SWIR Band 12 (2202.4 nm) is ideal for detecting molten lava because:
- Molten lava (700–1200 °C) emits strong thermal radiation in the SWIR range.
- B12 can detect thermal anomalies even when partially obscured by volcanic gases.
- Active lava flows often saturate B12, which makes detection straightforward.
- Cooling lava shows progressively lower B12 values, which makes intensity mapping possible.

**Thresholds:**
- The default thresholds are calibrated for typical volcanic eruptions.
- Adjust `lava_threshold` in the visualization function for the specific volcano and eruption intensity.
- Very active eruptions may need higher thresholds to separate the intensity levels.

## Attribution

The openEO-UDP process adapts wildfire visualization techniques to volcanic lava flow monitoring.

**Methodology:** SWIR Band 12 thermal emission detection, similar to the approaches in:
- Coppola, D., et al. (2016). Enhanced volcanic hot-spot detection using MODIS IR data.
- Wright, R., et al. (2004). MODVOLC: near-real-time thermal monitoring of global volcanism.